Finding low-altitude ARTEMIS passes and correlating with known lunar surface anomalies using SVM data from Tsunakawa et al. (2015). 

Start with the familiar 4 Dec 2013 THB data from Sawyer: 

In [5]:
import pyspedas

trange = ['2013-12-04/00:00', '2013-12-04/12:00']
loaded = pyspedas.projects.themis.state(probe='b', trange=trange)
print(loaded)   #is GSE position variable available?

22-Oct-26 17:16:10: File is current: themis_data/thb/l1/state/2013/thb_l1_state_20131204.cdf


['thb_pos', 'thb_vel', 'thb_spin_initial_delta_phi', 'thb_spin_idpu_spinper', 'thb_spin_segflags', 'thb_spin_ecl_initial_delta_phi', 'thb_spin_ecl_idpu_spinper', 'thb_spin_ecl_segflags', 'thb_pos_gse', 'thb_pos_gsm', 'thb_vel_gse', 'thb_vel_gsm', 'thb_pos_sel', 'thb_pos_sse', 'thb_vel_sel', 'thb_vel_sse']


File already containse SEL (moon-fixed selenographic position) for SVM, so do not need to convert again. Check units and centre quickly:

In [7]:
import numpy as np
from pyspedas import get_data, time_string

d = get_data('thb_pos_sel')
t, xyz = d.times, d.y
r = np.linalg.norm(xyz, axis=1)

print(xyz[:3]) #first few positions
print(r.min(), r.max()) #closest and furthest distance of approach
print(time_string(t[r.argmin()])) #time at which the closest approach occurs

[[-1399.0868    1502.8933     -14.914508]
 [-1281.8687    1547.2798     -14.522243]
 [-1161.8776    1588.2611     -14.074393]]
1752.193 18314.56
2013-12-04 00:12:00.000000


Closest approach is 1752 km at 00:12, so altitude is approx 14.8 km (consistent with Sawyer altitude plot). Furthest point is 18315 km for elliptical orbit.

In [8]:
print(np.diff(t[:4])) #check time between samples / s

alt = r - 1737.4
lat = np.degrees(np.arcsin(xyz[:, 2] / r))
lon = np.degrees(np.arctan2(xyz[:, 1], xyz[:, 0])) % 360

i = r.argmin()
print(lon[i], lat[i], alt[i]) #lowest sample lat, long and alt
print(time_string(t[alt < 100][[0, -1]]))  #first and last sample below 100 km

[60. 60. 60.]
85.109375 -0.25907627 14.792969
['2013-12-04 00:06:00.000000', '2013-12-04 00:18:00.000000']


Timestamps are UT; probe spent 12 minutes below 100 km from 00:06 to 00:18. Sampling at 1 position per minute.